# Fase 4: Modelado

**Pregunta experimental:** ¿qué relación tienen las características registradas de los pozos horizontales shale de la cuenca Neuquina con el volumen de agua inyectada, y qué capacidad predictiva muestran en operaciones posteriores? Esta fase usa la salida limpia de la fase 3 y se ejecuta desde la raíz del repositorio. La evaluación detallada corresponde a la fase 5.

| Elemento | Decisión |
|---|---|
| Alcance | Pozos horizontales shale de la cuenca Neuquina, seleccionados y preparados en la fase 3. |
| Objetivo | `agua_inyectada_m3`, volumen en m³; regresión supervisada, según la fase 2. |
| Entradas | Longitud de rama horizontal, cantidad de fracturas y tipo de terminación. La fase 3 las selecciona para el análisis histórico; el CSV no confirma que sus valores estuvieran disponibles antes de cada operación. |
| Algoritmo | `RandomForestRegressor`: aprende relaciones no lineales e interacciones sin exigir una fórmula previa. Su limitación es que no extrapola bien fuera de lo observado. |
| Referencia | `DummyRegressor` con mediana del entrenamiento. |
| Prueba | Operaciones iniciadas desde el 1/1/2025, reservadas sin usarlas para ajustar el modelo. |

Las columnas `profundidad_m`, `presion_inicial_psi` y `temperatura_c` no existen en el CSV original, según la comprobación documentada en la fase 3. No se usan `cuenca`, `tipo_reservorio` ni `subtipo_reservorio` porque son constantes tras el recorte. Tampoco se usa `formacion_productiva`: 2.630 de los 2.632 casos finales son Vaca Muerta, por lo que las otras dos categorías tienen respaldo insuficiente para aprender una diferencia fiable. No se usan identificadores, fechas de fin, duración, presión máxima ni cantidades efectivamente bombeadas como predictores: revelarían información de la operación terminada o facilitarían memorizar pozos. La comprobación temporal de la fase 3 no permite afirmar que `cantidad_fracturas` y `tipo_terminacion` fueran conocidas antes de la operación. El resultado se interpreta como evaluación retrospectiva sobre registros históricos. Para usarlo en planificación se necesitan valores planificados equivalentes; si no existen, habría que retirar esas variables y volver a entrenar.


## 4.1. Carga de los datos preparados

La fase 4 lee `df_limpio.csv`, generado por la fase 3 desde el CSV original versionado. El archivo ya contiene el recorte a pozos horizontales shale de la cuenca Neuquina y la exclusión de objetivos inválidos y fechas futuras al 28/10/2026. Se comprueban estas condiciones para detectar una salida vieja o equivocada. La selección de la población y la limpieza se documentan en la fase 3; aquí solo se preparan las columnas de entrada para el algoritmo.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from comun import exportar_modelos

RUTA_CSV = Path('df_limpio.csv')
if not RUTA_CSV.is_file():
    raise FileNotFoundError('Ejecutá primero la fase 3 desde la raíz del repositorio.')
df = pd.read_csv(RUTA_CSV)
df['fecha_inicio_fractura'] = pd.to_datetime(df['fecha_inicio_fractura'], errors='coerce')
print(f'Registros recibidos de fase 3: {len(df):,}')


In [ ]:
objetivo = 'agua_inyectada_m3'
entradas_numericas = ['longitud_rama_horizontal_m', 'cantidad_fracturas']
entradas_categoricas = ['tipo_terminacion']
entradas = entradas_numericas + entradas_categoricas
columnas_necesarias = entradas + [objetivo, 'cuenca', 'tipo_reservorio', 'subtipo_reservorio', 'idpozo', 'id_base_fractura_adjiv', 'fecha_inicio_fractura']
faltantes = set(columnas_necesarias) - set(df.columns)
if faltantes:
    raise ValueError(f'Columnas faltantes en la salida de fase 3: {sorted(faltantes)}')

alcance_correcto = (
    df['cuenca'].eq('NEUQUINA')
    & df['tipo_reservorio'].eq('NO CONVENCIONAL')
    & df['subtipo_reservorio'].eq('SHALE')
    & df['longitud_rama_horizontal_m'].gt(0)
)
agua = pd.to_numeric(df[objetivo], errors='coerce')
fecha_revision = pd.Timestamp('2026-10-28')
if not alcance_correcto.all() or not (agua.gt(0) & np.isfinite(agua)).all() or not df['fecha_inicio_fractura'].le(fecha_revision).all():
    raise ValueError('La salida de fase 3 no cumple el alcance o los criterios de calidad documentados; ejecutá de nuevo esa fase.')

datos_modelo = df[columnas_necesarias].copy()
datos_modelo[objetivo] = agua
for columna in entradas_numericas:
    datos_modelo[columna] = pd.to_numeric(datos_modelo[columna], errors='coerce').replace([np.inf, -np.inf], np.nan)
print(f'Registros listos para el modelo: {len(datos_modelo):,}')
print(datos_modelo[entradas + [objetivo]].isna().sum().to_frame('faltantes antes del pipeline'))


## 4.2. Separación temporal

El corte de 2025 deja las operaciones posteriores exclusivamente para prueba. Si un pozo figura en ambos períodos, se retira del entrenamiento para evitar que el modelo aprenda información del mismo pozo que luego se evalúa. El conjunto de prueba no participa de la imputación, codificación ni elección de parámetros.


In [ ]:
corte_temporal = pd.Timestamp('2025-01-01')
train = datos_modelo.loc[datos_modelo['fecha_inicio_fractura'] < corte_temporal].copy()
test = datos_modelo.loc[datos_modelo['fecha_inicio_fractura'] >= corte_temporal].copy()
pozos_compartidos = set(train['idpozo']).intersection(test['idpozo'])
train = train.loc[~train['idpozo'].isin(pozos_compartidos)].copy()
if train.empty or test.empty:
    raise ValueError('La partición temporal quedó vacía; revisar fechas y filtros.')
assert train['fecha_inicio_fractura'].max() < test['fecha_inicio_fractura'].min()
assert not set(train['idpozo']).intersection(test['idpozo'])
X_train, y_train = train[entradas], train[objetivo]
X_test, y_test = test[entradas], test[objetivo]
print(f'Entrenamiento: {len(train):,} operaciones; prueba: {len(test):,}.')
print(f'Pozos compartidos retirados del entrenamiento: {len(pozos_compartidos)}.')
print(f'Rango de entrenamiento: {train.fecha_inicio_fractura.min().date()} a {train.fecha_inicio_fractura.max().date()}')
print(f'Rango de prueba: {test.fecha_inicio_fractura.min().date()} a {test.fecha_inicio_fractura.max().date()}')


## 4.3. Entrenamiento

Las medianas numéricas y las categorías más frecuentes se calculan solo sobre entrenamiento. `OneHotEncoder` transforma las categorías y tolera categorías nuevas en prueba. Se fijan semilla y parámetros para obtener resultados reproducibles. `min_samples_leaf=3` limita hojas demasiado específicas; 200 árboles reducen la variación entre árboles con un costo razonable para este conjunto. Son parámetros iniciales, no se afirma que sean óptimos: su ajuste requeriría una validación temporal adicional dentro del período de entrenamiento.


In [ ]:
try:
    codificador = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:  # Compatibilidad con scikit-learn anterior a 1.2
    codificador = OneHotEncoder(handle_unknown='ignore', sparse=False)

prep = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), entradas_numericas),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', codificador),
    ]), entradas_categoricas),
])
modelo = Pipeline([
    ('prep', prep),
    ('regresor', RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42, n_jobs=-1)),
])
referencia = DummyRegressor(strategy='median')
modelo.fit(X_train, y_train)
referencia.fit(X_train, y_train)
print('Modelos entrenados. La prueba temporal permanece reservada para la fase 5.')


## 4.4. Salida para la fase 5

Se guardan el pipeline completo, la referencia, los datos de prueba y sus identificadores mediante `comun.exportar_modelos`. Los archivos en `artifacts/` son generados localmente al ejecutar esta notebook. La fase 5 deberá leer estos mismos artefactos y no reconstruir un conjunto de prueba distinto; sus cifras redactadas de antemano deben actualizarse con las métricas nuevas.


In [ ]:
exportar_modelos(
    modelo=modelo,
    referencia=referencia,
    X_test=X_test,
    y_test=y_test,
    test_metadata=test[['id_base_fractura_adjiv', 'idpozo', 'fecha_inicio_fractura']],
)
print(f'Artefactos guardados en {Path("artifacts").resolve()}')
